# 06 — Bedrock Model Benchmarking
Benchmarks Claude Haiku vs Sonnet latency, token consumption, and output
quality for the DataPilot agent roles. Results drive model routing decisions
in `model_id_registry.py` (FAST_AGENT_ROLES → Haiku, rest → Sonnet).

**Real-time integration**: The primary latency budget for WebSocket chat is
400ms to first token. This notebook validates that routing latency-sensitive
agents (Intent, SQL, Schema, Validation, Security) to Haiku keeps us within budget,
and that quality-sensitive agents (Insight, Critic, Planner) justify the Sonnet cost.

> **Note**: Cells that call the Bedrock API require valid AWS credentials.
> Run with `FEATURE_BEDROCK=true` and `AWS_PROFILE=datapilot-dev`.
> Cells marked `[MOCK]` run without credentials using MockLLMService.


In [ ]:
import sys, time, asyncio, json
sys.path.insert(0, '..')

# Model IDs
HAIKU  = 'anthropic.claude-haiku-4-5-20251001'
SONNET = 'anthropic.claude-sonnet-4-5'

# Pricing (USD per 1M tokens)
PRICING = {
    HAIKU:  {'input': 0.25,  'output': 1.25},
    SONNET: {'input': 3.00,  'output': 15.00},
}

def estimate_cost(model_id, input_tokens, output_tokens):
    p = PRICING.get(model_id, PRICING[SONNET])
    return round((input_tokens / 1e6) * p['input'] + (output_tokens / 1e6) * p['output'], 8)

print("Bedrock benchmarking notebook loaded.")
print(f"Models: Haiku={HAIKU} | Sonnet={SONNET}")


## [MOCK] Agent role latency simulation using MockLLMService

In [ ]:
# Uses MockLLMService — no AWS credentials required.
# Simulates realistic token counts and latency profiles for each agent role.
import random; random.seed(42)

AGENT_PROFILES = {
    #  agent_role        model    input_t  output_t  latency_ms  quality_score
    'intent':         (HAIKU,   250,  80,   200,  0.95),  # fast path
    'schema':         (HAIKU,   180,  20,   180,  0.93),  # single-word output
    'sql':            (HAIKU,   400, 120,   220,  0.91),  # SQL generation
    'validation':     (HAIKU,   600, 150,   240,  0.89),  # per-response check
    'security':       (HAIKU,   200,  60,   190,  0.97),  # safety critical
    'planner':        (SONNET, 800,  600,  1400,  0.96),  # plan quality critical
    'insight':        (SONNET,1200, 1000,  1800,  0.94),  # complex synthesis
    'critic':         (SONNET, 900,  500,  1300,  0.95),  # quality validation
    'recommendation': (SONNET,1000,  700,  1500,  0.93),  # business value
    'forecast':       (SONNET, 600,  300,  1200,  0.91),  # narrative generation
}

print(f"{'Agent':<18} {'Model':<10} {'Input':<8} {'Output':<8} {'Latency':<12} {'Cost/call':<12} {'Quality'}")
print("-" * 80)

total_haiku_cost  = 0
total_sonnet_cost = 0
websocket_budget_ok = True

for agent, (model, input_t, output_t, latency_ms, quality) in AGENT_PROFILES.items():
    cost = estimate_cost(model, input_t, output_t)
    model_short = 'Haiku' if model == HAIKU else 'Sonnet'
    if model == HAIKU:  total_haiku_cost  += cost
    else:               total_sonnet_cost += cost

    # WebSocket budget: Haiku agents must be < 300ms (they're on the hot path)
    budget_flag = ''
    if model == HAIKU and latency_ms > 300:
        budget_flag = ' ⚠ OVER BUDGET'
        websocket_budget_ok = False

    print(f"  {agent:<16} {model_short:<10} {input_t:<8} {output_t:<8} "
          f"{latency_ms:>6}ms     ${cost:<10.6f} {quality:.2f}{budget_flag}")

print(f"\nTotal cost per full pipeline run:")
print(f"  Haiku agents:  ${total_haiku_cost:.6f}")
print(f"  Sonnet agents: ${total_sonnet_cost:.6f}")
print(f"  Total:         ${total_haiku_cost + total_sonnet_cost:.6f}")
print(f"\nWebSocket budget (Haiku < 300ms): {'✓ OK' if websocket_budget_ok else '✗ OVER BUDGET'}")


## [MOCK] Cost comparison — Haiku routing saves vs Sonnet-only

In [ ]:
# What would we pay if all agents used Sonnet?
print("── Cost Comparison: FAST_AGENT_ROLES routing vs Sonnet-only ──\n")

sonnet_only_cost = 0
for agent, (model, input_t, output_t, latency_ms, quality) in AGENT_PROFILES.items():
    sonnet_only_cost += estimate_cost(SONNET, input_t, output_t)

mixed_cost = total_haiku_cost + total_sonnet_cost
savings    = sonnet_only_cost - mixed_cost
pct_saving = savings / sonnet_only_cost * 100

print(f"  Sonnet-only cost per pipeline:  ${sonnet_only_cost:.6f}")
print(f"  Mixed routing cost per pipeline: ${mixed_cost:.6f}")
print(f"  Savings per pipeline:            ${savings:.6f} ({pct_saving:.1f}%)")
print(f"\n  At 1,000 pipelines/day:")
print(f"    Sonnet-only: ${sonnet_only_cost*1000:.2f}/day")
print(f"    Mixed:       ${mixed_cost*1000:.2f}/day")
print(f"    Savings:     ${savings*1000:.2f}/day = ${savings*365000:.0f}/year")


## [MOCK] Streaming token delivery — first-token latency simulation

In [ ]:
# Simulates BedrockStreamAdapter token delivery timing
# Shows why streaming the executive summary improves perceived performance

async def simulate_streaming(total_tokens: int, first_token_ms: int, tokens_per_sec: int = 40):
    """Simulate token-by-token streaming from BedrockStreamAdapter."""
    events = []
    await asyncio.sleep(first_token_ms / 1000)  # wait for first token
    events.append(('first_token', first_token_ms))

    for i in range(1, total_tokens):
        await asyncio.sleep(1 / tokens_per_sec)
        events.append(('token', first_token_ms + int(i * 1000 / tokens_per_sec)))

    total_ms = first_token_ms + int(total_tokens * 1000 / tokens_per_sec)
    events.append(('complete', total_ms))
    return events

print("── Streaming vs Batch delivery — UX comparison ──\n")

scenarios = [
    ('Executive summary (streaming)', 150, 300, 40),   # first token in 300ms
    ('Executive summary (batch)',     150, 4500, 999),  # full response in 4.5s
    ('Insight headline (Haiku)',      30,  200, 40),    # fast, short
    ('SQL query (Haiku)',             50,  220, 40),    # short SQL string
]

for name, tokens, first_ms, tps in scenarios:
    events = asyncio.run(simulate_streaming(tokens, first_ms, min(tps, 999)))
    first  = next(e for e in events if e[0] == 'first_token')
    done   = next(e for e in events if e[0] == 'complete')
    print(f"  {name}")
    print(f"    User sees first content at: {first[1]}ms")
    print(f"    Full content ready at:      {done[1]}ms")
    ux = 'Excellent (streaming UX)' if first[1] < 400 else 'Poor (user waits)' if first[1] > 2000 else 'Acceptable'
    print(f"    UX rating:                  {ux}\n")


## Token tracker — validate cost attribution per session

In [ ]:
# Validate that TokenTracker correctly accumulates costs across agents

class MockTokenTracker:
    def __init__(self):
        self.usage = {}  # model_id → {input: int, output: int}

    def record(self, model_id: str, input_tokens: int, output_tokens: int):
        if model_id not in self.usage:
            self.usage[model_id] = {'input': 0, 'output': 0}
        self.usage[model_id]['input']  += input_tokens
        self.usage[model_id]['output'] += output_tokens

    def session_cost(self) -> float:
        total = 0.0
        for model_id, tokens in self.usage.items():
            total += estimate_cost(model_id, tokens['input'], tokens['output'])
        return round(total, 8)

    def summary(self) -> dict:
        return {
            model_id: {
                **tokens,
                'cost_usd': estimate_cost(model_id, tokens['input'], tokens['output'])
            }
            for model_id, tokens in self.usage.items()
        }

tracker = MockTokenTracker()
for agent, (model, input_t, output_t, _, _) in AGENT_PROFILES.items():
    tracker.record(model, input_t, output_t)

summary = tracker.summary()
print("── Session Token Usage Summary ──\n")
for model_id, data in summary.items():
    model_name = 'Haiku' if model_id == HAIKU else 'Sonnet'
    print(f"  {model_name}:")
    print(f"    Input tokens:  {data['input']:,}")
    print(f"    Output tokens: {data['output']:,}")
    print(f"    Cost:          ${data['cost_usd']:.6f}")

print(f"\n  Total session cost: ${tracker.session_cost():.6f}")
print(f"  Cost alert threshold: $0.10")
print(f"  Alert triggered: {'YES ⚠' if tracker.session_cost() > 0.10 else 'No ✓'}")


## [LIVE] Real Bedrock latency measurement (requires AWS credentials)

In [ ]:
# Uncomment and run this cell with real AWS credentials.
# Requires: AWS_PROFILE=datapilot-dev and model access approved in Bedrock console.

import os
BEDROCK_ENABLED = os.environ.get('FEATURE_BEDROCK', 'false').lower() == 'true'

if not BEDROCK_ENABLED:
    print("Skipping live Bedrock benchmark (FEATURE_BEDROCK not set).")
    print("To run: export FEATURE_BEDROCK=true && jupyter nbconvert --to notebook --execute 06_bedrock_model_benchmarking.ipynb")
else:
    from backend.infrastructure.llm.bedrock.bedrock_converse_adapter import BedrockConverseAdapter
    adapter = BedrockConverseAdapter()

    test_prompt = "Classify the intent: 'What is the total revenue by region this quarter?'"
    results = {}

    for model_id in [HAIKU, SONNET]:
        latencies = []
        for _ in range(5):  # 5 warm calls per model
            t0 = time.perf_counter()
            await adapter.complete(prompt=test_prompt, model_id=model_id, max_tokens=100)
            latencies.append((time.perf_counter() - t0) * 1000)

        results[model_id] = {
            'p50': sorted(latencies)[2],
            'p95': sorted(latencies)[4],
            'min': min(latencies),
            'max': max(latencies),
        }

    print(f"\n── Live Bedrock Latency (5 calls each) ──\n")
    for model_id, stats in results.items():
        name = 'Haiku' if model_id == HAIKU else 'Sonnet'
        print(f"  {name}:  P50={stats['p50']:.0f}ms  P95={stats['p95']:.0f}ms  "
              f"min={stats['min']:.0f}ms  max={stats['max']:.0f}ms")

    haiku_p50  = results[HAIKU]['p50']
    sonnet_p50 = results[SONNET]['p50']
    print(f"\n  Latency ratio (Sonnet/Haiku): {sonnet_p50/haiku_p50:.1f}x")
    print(f"  Haiku within 300ms budget: {'✓' if haiku_p50 < 300 else '✗'}")
